<a href="https://colab.research.google.com/github/madelsu/Dynamic_Hyperparameter_Optimization_LLM/blob/main/Gold_Standards(Clinical_Benchmarks)/GROUND_TRUTHS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# CELL 1 — SETUP: Mount Drive, Load Cohort CSV, Unzip Filtered Data
# =============================================================================
# NOTEBOOK : Ground Truth Scoring
# PURPOSE  : Load the cohort CSV and filtered Synthea data from Google Drive,
#            then load all clinical tables into memory as pandas DataFrames.
# INPUTS   : Google Drive → FINAL_FINAL_THESIS/DATA/
#              cohort_master.csv
#              Filtered_Study_Data.zip
# OUTPUTS  : cohort          — master patient DataFrame (in memory)
#            con_all         — all conditions
#            obs_all         — all observations/labs
#            med_all         — all medications
#            enc_all         — all encounters
#            proc_all        — all procedures
# PACKAGES : pandas, glob, zipfile, shutil, os, numpy
# =============================================================================

import os, glob, zipfile, shutil
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── 2. Paths ──────────────────────────────────────────────────────────────────
DRIVE_DATA    = '/content/drive/MyDrive/FINAL_FINAL_THESIS/Data'
COHORT_CSV    = f'{DRIVE_DATA}/cohort_master.csv'
SYNTHEA_ZIP   = f'{DRIVE_DATA}/Filtered_Study_Data.zip'
EXTRACT_DIR   = '/content/Filtered_Study_Data'
STUDY_END     = pd.Timestamp('2017-05-28')

# ── 3. Unzip Filtered_Study_Data ──────────────────────────────────────────────
print("📦 Extracting Filtered_Study_Data.zip ...")
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
    print("  🧹 Removed previous extraction")

with zipfile.ZipFile(SYNTHEA_ZIP, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

folders = sorted(glob.glob(f'{EXTRACT_DIR}/**/csv', recursive=True))
print(f"  ✅ Extracted {len(folders)} output folders")
for folder in folders:
    n = len(glob.glob(f'{folder}/*.csv'))
    print(f"     {os.path.basename(os.path.dirname(folder))} → {n} CSVs")

# ── 4. Load cohort CSV ────────────────────────────────────────────────────────
print("\n📋 Loading cohort_master.csv ...")
cohort = pd.read_csv(COHORT_CSV, low_memory=False)

DATE_COLS = ['Date_Diagnosis_Td', 'Date_Treatment_T0',
             'Date_T0_plus5', 'Date_T0_plus10',
             'BIRTHDATE', 'DEATHDATE', 'Observation_End', 'Global_Start_Date']
for col in DATE_COLS:
    if col in cohort.columns:
        cohort[col] = pd.to_datetime(cohort[col], errors='coerce')

# Derive covariate window boundaries for each index date
# Window = 5 years lookback ending on the index date itself
cohort['td_cov_start']  = cohort['Date_Diagnosis_Td']  - pd.DateOffset(years=5)
cohort['td_cov_end']    = cohort['Date_Diagnosis_Td']
cohort['t0_cov_start']  = cohort['Date_Treatment_T0']  - pd.DateOffset(years=5)
cohort['t0_cov_end']    = cohort['Date_Treatment_T0']
cohort['t5_cov_start']  = cohort['Date_T0_plus5']      - pd.DateOffset(years=5)
cohort['t5_cov_end']    = cohort['Date_T0_plus5']
cohort['t10_cov_start'] = cohort['Date_T0_plus10']     - pd.DateOffset(years=5)
cohort['t10_cov_end']   = cohort['Date_T0_plus10']

# Eligibility flags
cohort['eligible_t5']  = (cohort['Observation_End'] >= cohort['Date_T0_plus5']).astype(int)
cohort['eligible_t10'] = (cohort['Observation_End'] >= cohort['Date_T0_plus10']).astype(int)

print(f"  ✅ Cohort loaded: {len(cohort):,} patients")
print(f"     Eligible T5  : {cohort['eligible_t5'].sum():,}")
print(f"     Eligible T10 : {cohort['eligible_t10'].sum():,}")

# ── 5. Load clinical tables ───────────────────────────────────────────────────
SYNTHEA_PATTERN = f'{EXTRACT_DIR}/output_*/csv'
print("\n📂 Loading clinical tables ...")

def load_table(table_name, date_cols=None):
    files = sorted(glob.glob(f'{SYNTHEA_PATTERN}/{table_name}.csv'))
    if not files:
        print(f"  ⚠️  {table_name}: no files found")
        return pd.DataFrame()
    df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
    if 'PATIENT' in df.columns:
        df['PATIENT'] = df['PATIENT'].astype(str).str.strip()
    for dc in (date_cols or []):
        if dc in df.columns:
            df[dc] = pd.to_datetime(df[dc], errors='coerce')
    print(f"  ✅ {table_name:<16} {len(df):>9,} rows  ({len(files)} shards)")
    return df

con_all  = load_table('conditions',   ['START'])
obs_all  = load_table('observations', ['DATE'])
med_all  = load_table('medications',  ['START'])
enc_all  = load_table('encounters',   ['DATE'])
proc_all = load_table('procedures',   ['DATE'])

# Standardise CODE to string for LOINC matching in observations
obs_all['CODE'] = obs_all['CODE'].astype(str).str.strip()
# Standardise CODE to numeric for SNOMED matching in conditions/medications
for df in [con_all, med_all]:
    df['CODE'] = pd.to_numeric(df['CODE'], errors='coerce')

print(f"\n✅ Setup complete — ready for ground truth scoring")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 Extracting Filtered_Study_Data.zip ...
  ✅ Extracted 12 output folders
     output_1 → 0 CSVs
     output_10 → 9 CSVs
     output_11 → 9 CSVs
     output_12 → 9 CSVs
     output_2 → 9 CSVs
     output_3 → 9 CSVs
     output_4 → 0 CSVs
     output_5 → 9 CSVs
     output_6 → 9 CSVs
     output_7 → 9 CSVs
     output_8 → 9 CSVs
     output_9 → 0 CSVs

📋 Loading cohort_master.csv ...
  ✅ Cohort loaded: 46,330 patients
     Eligible T5  : 41,913
     Eligible T10 : 36,934

📂 Loading clinical tables ...
  ✅ conditions         369,487 rows  (9 shards)
  ✅ observations     4,764,102 rows  (9 shards)
  ✅ medications        269,133 rows  (9 shards)
  ✅ encounters         416,220 rows  (9 shards)
  ✅ procedures         211,046 rows  (9 shards)

✅ Setup complete — ready for ground truth scoring


📂 Loading careplans table...
  ✅ careplans          451,145 rows  (9 shards)
   careplans: 451,145 rows

DOMAIN                          W  SPEC_CODES   FOUND_IN_CONDITIONS   FOUND_IN_PROCEDURES   FOUND_IN_ENC/CARE
  neuropathy                    1           4                     1                     0                   0  
  foot_ulcer                    1           3                     0                     0                   0  ❌ ZERO HITS ANYWHERE
  gangrene                      2           1                     0                     0                   0  ❌ ZERO HITS ANYWHERE
  amputation                    3           3                     0                     1                   0  
  retinopathy                   1           6                     3                     0                   0  
  laser_therapy                 2           2                     0                     1                   0  
  blindness                     2           4                     1      

In [ ]:
# =============================================================================
# CELL 2 — GROUND TRUTH SCORING (Young lab-aug, Young codes-only, Zghebi DISSCO, Cooper)
# =============================================================================
# NOTEBOOK : Ground Truth Scoring
# PURPOSE  : Score all four GT algorithms across all four index dates
#            (Td, T0, T5, T10) and add results as new columns to cohort.
# INPUTS   : cohort, con_all, obs_all, med_all  (from Cell 1)
# OUTPUTS  : cohort — updated with GT tier + evidence columns
#            /content/cohort_with_GT.csv
# COLUMNS ADDED (per prefix td/t0/t5/t10, per GT young/dissco/cooper):
#   {prefix}_young_tier        — Baseline T2D / Mild / Moderate / Advanced/Critical
#   {prefix}_young_evidence    — domain scores and lab values used
#   {prefix}_young_codes_tier  — same but codes-only (no labs), sensitivity analysis
#   {prefix}_dissco_score      — raw weighted DISSCO score
#   {prefix}_dissco_tier       — 4-tier classification
#   {prefix}_dissco_evidence   — domains triggered and their weights
#   {prefix}_cooper_tier       — Baseline T2D / Mild / Moderate / Advanced/Critical
#   {prefix}_cooper_evidence   — 7-dimension evidence string
# PACKAGES : pandas, numpy
# =============================================================================

import pandas as pd
import numpy as np

# ══════════════════════════════════════════════════════════════════════════════
# SHARED HELPER
# ══════════════════════════════════════════════════════════════════════════════
def slice_window(df, date_col, start_col, end_col, ids):
    """Filter df to rows within each patient's own time window."""
    d = df[df['PATIENT'].isin(ids)].copy()
    return d[(d[date_col] >= d[start_col]) & (d[date_col] <= d[end_col])]

def attach_windows(df, cohort_sub, window_prefix):
    """Merge patient-specific window boundaries onto a clinical dataframe."""
    win = cohort_sub[['PATIENT',
                       f'{window_prefix}_cov_start',
                       f'{window_prefix}_cov_end']].copy()
    return df.merge(win, on='PATIENT', how='inner')

# ══════════════════════════════════════════════════════════════════════════════
# GT 1 & 2 — YOUNG ET AL. (lab-augmented PRIMARY + codes-only SENSITIVITY)
# Reference: Young et al. 2008 + Glasheen 2017 ICD-10 update → SNOMED mapping
# ══════════════════════════════════════════════════════════════════════════════

# SNOMED codes per domain and severity level
YOUNG_DOMAINS = {
    'retinopathy': {
        'level2': {4855003, 193349004, 232020009, 60951000119105},  # proliferative / blindness
        'level1': {193350004, 193387007, 232019003, 399863005,       # NPDR / background
                   422034002, 1551000119108, 157141000119108},
    },
    'nephropathy': {
        'level2': {46177005, 175901007, 428575007, 236435004},       # ESRD / dialysis
        'level1': {127013003, 431856006, 90688005, 709044004,        # diabetic nephropathy / CKD
                   90781000119102},                                   # microalbuminuria
    },
    'neuropathy': {     # binary — no level2
        'level1': {230572002, 299960009, 140101000119109, 57764000,
                   60951000119105, 368581000119106},
    },
    'pvd': {
        'level2': {372070002, 57171000119105, 271651004},            # gangrene / amputation
        'level1': {399957001, 40275004, 161695004, 698754002},       # PAD / claudication
    },
    'stroke': {
        'level2': {230690007, 422504002, 413758000},                 # stroke
        'level1': {266257000, 230716006},                            # TIA
    },
    'cvd': {
        'level2': {22298006, 57054005, 394659003, 233829000},        # MI / ACS
        'level1': {194828000, 53741008, 84114007},                   # angina / CHD / HF
    },
    'metabolic': {
        'level2': {420422005, 719216001},                            # DKA / HHS
        'level1': {302866003, 421725003, 237636009},                 # severe hypoglycaemia
    },
}

# LOINC codes for lab augmentation
LOINC_CREAT = '2160-0'    # serum creatinine — use MAX (worst)
LOINC_UACR  = '14959-1'   # UACR — use MAX (worst)
LOINC_HBA1C = '4548-4'    # HbA1c — use MAX

def _get_lab_max(obs_slice, loinc_code):
    """Return Series {PATIENT: max_value} for a given LOINC within the slice."""
    sub = (obs_slice[obs_slice['CODE'] == loinc_code]
           .copy()
           .assign(VAL=lambda x: pd.to_numeric(x['VALUE'], errors='coerce'))
           .dropna(subset=['VAL'])
           .query('VAL > 0'))
    return sub.groupby('PATIENT')['VAL'].max() if len(sub) else pd.Series(dtype=float)

def compute_young(con_slice, obs_slice, patient_ids, prefix, lab_augmented=True):
    """
    Score Young DCSI for each patient.

    Parameters
    ----------
    lab_augmented : if True → primary GT (uses creatinine + UACR for nephropathy)
                    if False → codes-only sensitivity analysis
    """
    # Pre-compute which patients have each domain code
    patient_codes = (con_slice.groupby('PATIENT')['CODE']
                               .apply(set)
                               .reindex(patient_ids, fill_value=set()))

    # Lab series (only computed when needed)
    if lab_augmented:
        creat_max = _get_lab_max(obs_slice, LOINC_CREAT)
        uacr_max  = _get_lab_max(obs_slice, LOINC_UACR)
    hba1c_max = _get_lab_max(obs_slice, LOINC_HBA1C)

    rows = []
    for pid in patient_ids:
        codes = patient_codes.get(pid, set())
        domain_scores = {}

        for domain, levels in YOUNG_DOMAINS.items():
            if domain == 'neuropathy':        # binary
                score = 1 if codes & levels['level1'] else 0
            else:
                if codes & levels.get('level2', set()):
                    score = 2
                elif codes & levels.get('level1', set()):
                    score = 1
                else:
                    score = 0

            # Lab augmentation for nephropathy
            if domain == 'nephropathy' and lab_augmented:
                creat = creat_max.get(pid, np.nan)
                uacr  = uacr_max.get(pid,  np.nan)
                lab_score = 0
                if pd.notna(creat):
                    lab_score = 2 if creat > 2.0 else (1 if creat >= 1.5 else 0)
                if pd.notna(uacr) and uacr >= 300:
                    lab_score = max(lab_score, 1)
                score = max(score, lab_score)

            domain_scores[domain] = score

        total = sum(domain_scores.values())
        hba1c = hba1c_max.get(pid, np.nan)

        # Tier assignment
        if   total == 0: tier = 'Baseline T2D'
        elif total == 1: tier = 'Mild Complications'
        elif total <= 3: tier = 'Moderate Complications'
        else:            tier = 'Advanced/Critical'

        # Evidence string — only non-zero domains + lab values
        ev_parts = [f"{d}={s}" for d, s in domain_scores.items() if s > 0]
        if lab_augmented:
            creat = creat_max.get(pid, np.nan)
            uacr  = uacr_max.get(pid, np.nan)
            if pd.notna(creat): ev_parts.append(f"creatinine={creat:.2f}mg/dL")
            if pd.notna(uacr):  ev_parts.append(f"uacr={uacr:.1f}mg/g")
        if pd.notna(hba1c):     ev_parts.append(f"hba1c={hba1c:.1f}%")
        evidence = '; '.join(ev_parts) if ev_parts else 'no complications detected'

        rows.append({
            'PATIENT':                   pid,
            f'{prefix}_dcsi_total':      total,
            **{f'{prefix}_young_{d}': s for d, s in domain_scores.items()},
            f'{prefix}_young_tier':      tier,
            f'{prefix}_young_evidence':  evidence,
        })

    return pd.DataFrame(rows)

# ══════════════════════════════════════════════════════════════════════════════
# GT 3 — ZGHEBI ET AL. DISSCO (weighted sum of 34 domains)
# Reference: Zghebi et al. 2020, BMJ Open Diabetes Research & Care
# ══════════════════════════════════════════════════════════════════════════════

DISSCO_DOMAINS = {
    # (domain_name, snomed_codes_set, weight)
    # Group A — Diabetes complications & renal
    'neuropathy':           ({'codes': {230572002, 299960009, 140101000119109, 368581000119106}, 'weight': 1}),
    'foot_ulcer':           ({'codes': {161695004, 57177007, 47801003},          'weight': 1}),
    'gangrene':             ({'codes': {372070002},                               'weight': 2}),
    'amputation':           ({'codes': {57171000119105, 271651004, 180030006},    'weight': 3}),
    'retinopathy':          ({'codes': {193350004, 193387007, 232019003, 422034002,
                                        1551000119108, 157141000119108},          'weight': 1}),
    'laser_therapy':        ({'codes': {415070008, 406149000},                    'weight': 2}),  # procedures
    'blindness':            ({'codes': {397540003, 193349004, 4855003, 60951000119105}, 'weight': 2}),
    'diabetic_nephropathy': ({'codes': {127013003, 431856006, 90688005, 90781000119102}, 'weight': 1}),
    'albuminuria':          ({'codes': {34436003, 157141000119108},               'weight': 2, 'uacr_threshold': 30}),
    'esrd':                 ({'codes': {46177005, 175901007, 428575007},          'weight': 3}),
    'hypoglycaemia':        ({'codes': {302866003, 421725003, 237636009},         'weight': 1}),
    'dka':                  ({'codes': {420422005},                               'weight': 2}),
    'hhs':                  ({'codes': {719216001},                               'weight': 3}),
    # Group B — Cardiovascular
    'hypertension':         ({'codes': {38341003, 59621000},                      'weight': 0.5}),
    'hyperlipidaemia':      ({'codes': {55822004, 13644009},                      'weight': 0.5}),
    'stable_angina':        ({'codes': {194828000, 57822003},                     'weight': 1}),
    'mi_acs':               ({'codes': {22298006, 57054005, 394659003, 233829000}, 'weight': 2}),
    'heart_valve':          ({'codes': {368009, 79619009},                        'weight': 1}),
    'endocarditis':         ({'codes': {56819008},                                'weight': 2}),
    'coronary_intervention':({'codes': {11101003, 36969009},                      'weight': 2}),
    'cardiac_arrest':       ({'codes': {410429000},                               'weight': 3}),
    'afib':                 ({'codes': {49436004, 6456007},                       'weight': 1}),
    'heart_failure':        ({'codes': {84114007, 42343007},                      'weight': 3}),
    'pvd':                  ({'codes': {399957001, 40275004, 698754002},          'weight': 1}),
    'cardiomyopathy':       ({'codes': {57772001, 85898001},                      'weight': 1}),
    'chd':                  ({'codes': {53741008, 414545008},                     'weight': 1}),
    # Group C — Cerebrovascular
    'tia':                  ({'codes': {266257000, 230716006},                    'weight': 2}),
    'stroke':               ({'codes': {230690007, 422504002, 413758000},         'weight': 4}),
}

def compute_dissco(con_slice, obs_slice, proc_slice, patient_ids, prefix):
    """
    Score Zghebi DISSCO for each patient.
    Combines condition codes, procedure codes, and UACR lab for domain 9.
    """
    # Combine conditions + procedures for code lookup
    all_codes_df = pd.concat([
        con_slice[['PATIENT', 'CODE']],
        proc_slice[['PATIENT', 'CODE']],
    ], ignore_index=True)
    patient_codes = (all_codes_df.groupby('PATIENT')['CODE']
                                  .apply(set)
                                  .reindex(patient_ids, fill_value=set()))

    uacr_max = _get_lab_max(obs_slice, LOINC_UACR)

    rows = []
    for pid in patient_ids:
        codes        = patient_codes.get(pid, set())
        uacr         = uacr_max.get(pid, np.nan)
        total_weight = 0.0
        triggered    = []

        for domain_name, info in DISSCO_DOMAINS.items():
            domain_codes = info['codes']
            weight       = info['weight']
            hit = bool(codes & domain_codes)

            # Domain 9 (albuminuria) also triggered by UACR lab
            if domain_name == 'albuminuria':
                uacr_threshold = info.get('uacr_threshold', 30)
                if pd.notna(uacr) and uacr >= uacr_threshold:
                    hit = True

            if hit:
                total_weight += weight
                triggered.append(f"{domain_name}(w={weight})")

        # Tier assignment — provisional thresholds from Zghebi 2020 adaptation
        if   total_weight == 0: tier = 'Baseline T2D'
        elif total_weight <= 3: tier = 'Mild Complications'
        elif total_weight <= 8: tier = 'Moderate Complications'
        else:                   tier = 'Advanced/Critical'

        evidence = '; '.join(triggered) if triggered else 'no qualifying domain'

        rows.append({
            'PATIENT':                   pid,
            f'{prefix}_dissco_score':    round(total_weight, 1),
            f'{prefix}_dissco_tier':     tier,
            f'{prefix}_dissco_evidence': evidence,
        })

    return pd.DataFrame(rows)

# ══════════════════════════════════════════════════════════════════════════════
# GT 4 — COOPER ET AL. (7-dimension priority-rule classifier)
# Reference: Cooper et al. 2025, JBI — green-rated phenotypes only
# ══════════════════════════════════════════════════════════════════════════════

COOPER_COMPLICATIONS = {
    'Nephropathy/CKD':  [127013003, 90781000119102, 46177005, 709044004],
    'Retinopathy':      [422034002, 1551000119108, 157141000119108, 97331000119101, 1501000119109],
    'Neuropathy':       [60951000119105, 368581000119106, 199049005],
    'Foot/Amputation':  [37130007, 129331000119104, 57171000119105, 161695004],
    'Stroke/CBV':       [230690007, 266257000],
    'Coronary/MI':      [22298006, 233829000, 194828000, 53741008],
    'Heart_failure':    [84114007],
    'Peripheral_vasc':  [698754002],
}
RET_IDS = {
    3: [97331000119101, 1501000119109],
    2: [1551000119108, 157141000119108],
    1: [422034002],
}
RET_NAMES   = {0: 'None', 1: 'Background', 2: 'NPDR', 3: 'Proliferative/Edema'}
LOINC_EGFR  = '33914-3'
DRUG_CLASSES = {
    'Metformin':    [860975, 860978, 860981, 860996, 861004, 861006, 861010],
    'Insulin':      [847230, 847207, 847209, 847257, 106892],
    'Sulfonylurea': [316049, 197805],
    'Liraglutide':  [897122],
    'Canagliflozin':[1373463],
}

def compute_cooper(con_slice, obs_slice, med_slice, patient_ids, prefix):
    """Score Cooper et al. 7-dimension phenotype for each patient."""
    def pts_cond(cids):
        m = con_slice['CODE'].isin(cids)
        return set(con_slice[m]['PATIENT'].unique())

    def pts_med(cids):
        m = med_slice['CODE'].isin(cids)
        return set(med_slice[m]['PATIENT'].unique())

    egfr_min  = _get_lab_max(obs_slice, LOINC_EGFR)    # min eGFR = worst
    # override: use min not max for eGFR
    sub_egfr = (obs_slice[obs_slice['CODE'] == LOINC_EGFR]
                .assign(VAL=lambda x: pd.to_numeric(x['VALUE'], errors='coerce'))
                .dropna(subset=['VAL']).query('VAL > 0'))
    egfr_min  = sub_egfr.groupby('PATIENT')['VAL'].min() if len(sub_egfr) else pd.Series(dtype=float)
    uacr_max  = _get_lab_max(obs_slice, LOINC_UACR)
    hba1c_max = _get_lab_max(obs_slice, LOINC_HBA1C)

    cond_sets = {k: pts_cond(v) for k, v in COOPER_COMPLICATIONS.items()}
    ret_sets  = {k: pts_cond(v) for k, v in RET_IDS.items()}
    med_sets  = {k: pts_med(v)  for k, v in DRUG_CLASSES.items()}

    rows = []
    for pid in patient_ids:
        comps      = sum(1 for s in cond_sets.values() if pid in s)
        uacr       = uacr_max.get(pid, np.nan)
        egfr       = egfr_min.get(pid,  np.nan)
        hba1c      = hba1c_max.get(pid, np.nan)
        uacr_cat   = ('Unknown'          if pd.isna(uacr)  else
                      'Normal'           if uacr < 30      else
                      'Microalbuminuria' if uacr < 300     else 'Macroalbuminuria')
        ret_stage  = (3 if pid in ret_sets[3] else
                      2 if pid in ret_sets[2] else
                      1 if pid in ret_sets[1] else 0)
        n_meds     = sum(1 for s in med_sets.values() if pid in s)
        on_insulin = int(pid in med_sets['Insulin'])
        has_foot   = int(pid in cond_sets['Foot/Amputation'])
        has_macro  = int(any(pid in cond_sets[k]
                             for k in ['Stroke/CBV','Coronary/MI','Heart_failure','Peripheral_vasc']))
        has_micro  = int(
            any(pid in cond_sets[k] for k in ['Nephropathy/CKD','Retinopathy','Neuropathy']) or
            uacr_cat in ['Microalbuminuria','Macroalbuminuria'] or
            (pd.notna(egfr) and egfr < 60)
        )

        if comps >= 3 or has_foot or ret_stage == 3 or (pd.notna(egfr) and egfr < 30):
            tier = 'Advanced/Critical'
        elif has_macro or has_micro:
            tier = 'Moderate Complications'
        elif (pd.notna(hba1c) and hba1c > 8.0) or n_meds > 1:
            tier = 'Mild Complications'
        else:
            tier = 'Baseline T2D'

        ev = [f"comp_count={comps}", f"uacr={uacr_cat}", f"retinopathy={RET_NAMES[ret_stage]}"]
        if on_insulin:          ev.append('insulin=True')
        if has_foot:            ev.append('foot_ulcer=True')
        if pd.notna(egfr):      ev.append(f'egfr={egfr:.1f}mL/min')
        if pd.notna(hba1c):     ev.append(f'hba1c={hba1c:.1f}%')
        if has_macro:           ev.append('macrovascular=True')
        if has_micro:           ev.append('microvascular=True')
        if n_meds > 0:          ev.append(f'n_drug_classes={n_meds}')

        rows.append({
            'PATIENT':                   pid,
            f'{prefix}_cooper_tier':     tier,
            f'{prefix}_cooper_evidence': '; '.join(ev),
        })

    return pd.DataFrame(rows)
# ══════════════════════════════════════════════════════════════════════════════
# GT 3 — ZGHEBI ET AL. DISSCO (weighted sum of 34 domains)
# Reference: Zghebi et al. 2020, BMJ Open Diabetes Research & Care
# ══════════════════════════════════════════════════════════════════════════════

DISSCO_DOMAINS = {
    # domain_name: {'codes': set_of_snomed_ints, 'weight': float}
    # Group A — Diabetes complications & renal
    'neuropathy':            {'codes': {230572002, 299960009, 140101000119109, 368581000119106}, 'weight': 1},
    'foot_ulcer':            {'codes': {161695004, 57177007, 47801003},                          'weight': 1},
    'gangrene':              {'codes': {372070002},                                               'weight': 2},
    'amputation':            {'codes': {57171000119105, 271651004, 180030006},                    'weight': 3},
    'retinopathy':           {'codes': {193350004, 193387007, 232019003, 422034002,
                                        1551000119108, 157141000119108},                          'weight': 1},
    'laser_therapy':         {'codes': {415070008, 406149000},                                    'weight': 2},
    'blindness':             {'codes': {397540003, 193349004, 4855003, 60951000119105},           'weight': 2},
    'diabetic_nephropathy':  {'codes': {127013003, 431856006, 90688005, 90781000119102},          'weight': 1},
    'albuminuria':           {'codes': {34436003, 157141000119108}, 'weight': 2, 'uacr_threshold': 30},
    'esrd':                  {'codes': {46177005, 175901007, 428575007},                          'weight': 3},
    'hypoglycaemia':         {'codes': {302866003, 421725003, 237636009},                         'weight': 1},
    'dka':                   {'codes': {420422005},                                               'weight': 2},
    'hhs':                   {'codes': {719216001},                                               'weight': 3},
    # Group B — Cardiovascular
    'hypertension':          {'codes': {38341003, 59621000},                                      'weight': 0.5},
    'hyperlipidaemia':       {'codes': {55822004, 13644009},                                      'weight': 0.5},
    'stable_angina':         {'codes': {194828000, 57822003},                                     'weight': 1},
    'mi_acs':                {'codes': {22298006, 57054005, 394659003, 233829000},                'weight': 2},
    'heart_valve':           {'codes': {368009, 79619009},                                        'weight': 1},
    'endocarditis':          {'codes': {56819008},                                                'weight': 2},
    'coronary_intervention': {'codes': {11101003, 36969009},                                      'weight': 2},
    'cardiac_arrest':        {'codes': {410429000},                                               'weight': 3},
    'afib':                  {'codes': {49436004, 6456007},                                       'weight': 1},
    'heart_failure':         {'codes': {84114007, 42343007},                                      'weight': 3},
    'pvd':                   {'codes': {399957001, 40275004, 698754002},                          'weight': 1},
    'cardiomyopathy':        {'codes': {57772001, 85898001},                                      'weight': 1},
    'chd':                   {'codes': {53741008, 414545008},                                     'weight': 1},
    # Group C — Cerebrovascular
    'tia':                   {'codes': {266257000, 230716006},                                    'weight': 2},
    'stroke':                {'codes': {230690007, 422504002, 413758000},                         'weight': 4},
}

LOINC_UACR_DISSCO = "14959-1"   # same LOINC as Young/Cooper — reuse


def compute_dissco(con_slice, obs_slice, proc_slice, patient_ids, prefix):
    """
    Score Zghebi DISSCO for each patient.
    Combines condition codes + procedure codes for domain lookup,
    and UACR lab value for the albuminuria domain.

    Parameters
    ----------
    con_slice   : windowed conditions DataFrame  (columns: PATIENT, CODE)
    obs_slice   : windowed observations DataFrame (columns: PATIENT, CODE, VALUE)
    proc_slice  : windowed procedures DataFrame  (columns: PATIENT, CODE)
    patient_ids : iterable of patient ID strings
    prefix      : str, e.g. "t5" → output columns t5_dissco_score, etc.

    Returns
    -------
    DataFrame with columns:
        PATIENT, {prefix}_dissco_score, {prefix}_dissco_tier,
        {prefix}_dissco_evidence
    """
    patient_ids = list(patient_ids)

    # ── Build per-patient code sets (conditions + procedures combined) ────────
    all_codes_df = pd.concat([
        con_slice[['PATIENT', 'CODE']].copy(),
        proc_slice[['PATIENT', 'CODE']].copy(),
    ], ignore_index=True)
    all_codes_df['CODE'] = pd.to_numeric(all_codes_df['CODE'], errors='coerce')
    patient_codes = (
        all_codes_df.dropna(subset=['CODE'])
        .groupby('PATIENT')['CODE']
        .apply(set)
        .reindex(patient_ids, fill_value=set())
    )

    # ── UACR max per patient (reuses same LOINC as other GTs) ─────────────────
    obs_slice = obs_slice.copy()
    obs_slice['VALUE_NUM'] = pd.to_numeric(obs_slice['VALUE'], errors='coerce')
    uacr_sub = (obs_slice[obs_slice['CODE'].astype(str) == LOINC_UACR_DISSCO]
                .dropna(subset=['VALUE_NUM'])
                .query('VALUE_NUM > 0'))
    uacr_max = (uacr_sub.groupby('PATIENT')['VALUE_NUM'].max()
                if len(uacr_sub) else pd.Series(dtype=float))

    # ── Score each patient ────────────────────────────────────────────────────
    rows = []
    for pid in patient_ids:
        codes        = patient_codes.get(pid, set())
        uacr         = uacr_max.get(pid, np.nan)
        total_weight = 0.0
        triggered    = []

        for domain_name, info in DISSCO_DOMAINS.items():
            domain_codes = info['codes']
            weight       = info['weight']
            hit          = bool(codes & domain_codes)

            # Albuminuria domain: also triggered by UACR lab ≥ threshold
            if domain_name == 'albuminuria':
                threshold = info.get('uacr_threshold', 30)
                if pd.notna(uacr) and uacr >= threshold:
                    hit = True

            if hit:
                total_weight += weight
                triggered.append(f"{domain_name}(w={weight})")

        # ── Tier assignment (four-tier mapping consistent with Young/Cooper) ──
        if   total_weight == 0:  tier = 'Baseline T2D'
        elif total_weight <= 3:  tier = 'Mild Complications'
        elif total_weight <= 8:  tier = 'Moderate Complications'
        else:                    tier = 'Advanced/Critical'

        evidence = '; '.join(triggered) if triggered else 'no qualifying domain'

        rows.append({
            'PATIENT':                    pid,
            f'{prefix}_dissco_score':     round(total_weight, 1),
            f'{prefix}_dissco_tier':      tier,
            f'{prefix}_dissco_evidence':  evidence,
        })

    return pd.DataFrame(rows)
# ══════════════════════════════════════════════════════════════════════════════
# SCORE ALL FOUR INDEX DATES
# ══════════════════════════════════════════════════════════════════════════════

eligible_ids = set(cohort['PATIENT'])
ids_t5       = set(cohort.loc[cohort['eligible_t5']  == 1, 'PATIENT'])
ids_t10      = set(cohort.loc[cohort['eligible_t10'] == 1, 'PATIENT'])

# (prefix, ids, window_start_col, window_end_col)
WINDOWS = [
    ('td',  eligible_ids, 'td_cov_start',  'td_cov_end'),
    ('t0',  eligible_ids, 't0_cov_start',  't0_cov_end'),
    ('t5',  ids_t5,       't5_cov_start',  't5_cov_end'),
    ('t10', ids_t10,      't10_cov_start', 't10_cov_end'),
]

print("=" * 60)
print("GROUND TRUTH SCORING — ALL INDEX DATES")
print("=" * 60)

for prefix, ids, start_col, end_col in WINDOWS:
    print(f"\n── {prefix.upper()} ({len(ids):,} patients) ──────────────────")

    # Attach patient-specific window dates to clinical tables
    win_cols = cohort[['PATIENT', start_col, end_col]].rename(
        columns={start_col: 'WIN_START', end_col: 'WIN_END'})

    def slice_w(df, date_col):
        d = df[df['PATIENT'].isin(ids)].merge(win_cols, on='PATIENT', how='inner')
        return d[(d[date_col] >= d['WIN_START']) & (d[date_col] <= d['WIN_END'])]

    con_sl  = slice_w(con_all,  'START')
    obs_sl  = slice_w(obs_all,  'DATE')
    med_sl  = slice_w(med_all,  'START')
    proc_sl = slice_w(proc_all, 'DATE')
    print(f"   Sliced: cond={len(con_sl):,} | obs={len(obs_sl):,} | "
          f"med={len(med_sl):,} | proc={len(proc_sl):,}")

    # Young GT — lab-augmented (primary)
    y_lab = compute_young(con_sl, obs_sl, list(ids), prefix, lab_augmented=True)
    print(f"   Young (lab-aug):\n{y_lab[f'{prefix}_young_tier'].value_counts().to_string()}")

    # Young GT — codes-only (sensitivity analysis)
    y_codes = compute_young(con_sl, obs_sl, list(ids), prefix+'_codes', lab_augmented=False)
    y_codes = y_codes.rename(columns={
        f'{prefix}_codes_young_tier':     f'{prefix}_young_codes_tier',
        f'{prefix}_codes_young_evidence': f'{prefix}_young_codes_evidence',
        f'{prefix}_codes_dcsi_total':     f'{prefix}_young_codes_dcsi_total',
    })
    # Keep only tier + evidence columns for codes-only (domain scores already in lab-aug)
    y_codes = y_codes[['PATIENT',
                        f'{prefix}_young_codes_tier',
                        f'{prefix}_young_codes_evidence',
                        f'{prefix}_young_codes_dcsi_total']]

    # Zghebi DISSCO
    d_scores = compute_dissco(con_sl, obs_sl, proc_sl, list(ids), prefix)
    print(f"   DISSCO:\n{d_scores[f'{prefix}_dissco_tier'].value_counts().to_string()}")

    # Cooper GT
    c_scores = compute_cooper(con_sl, obs_sl, med_sl, list(ids), prefix)
    print(f"   Cooper:\n{c_scores[f'{prefix}_cooper_tier'].value_counts().to_string()}")

    # Merge all GT results into cohort
    for df_result in [y_lab, y_codes, d_scores, c_scores]:
        # Drop any columns already present to avoid _x/_y duplicates on re-run
        existing = [c for c in df_result.columns if c in cohort.columns and c != 'PATIENT']
        cohort = cohort.drop(columns=existing, errors='ignore')
        cohort = cohort.merge(df_result, on='PATIENT', how='left')

print(f"\n✅ All GT scores merged — cohort shape: {cohort.shape}")

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("GT TIER DISTRIBUTION SUMMARY")
print("=" * 60)

TIER_ORDER = ['Baseline T2D', 'Mild Complications',
              'Moderate Complications', 'Advanced/Critical']

for prefix, ids, _, _ in WINDOWS:
    print(f"\n  {prefix.upper()}  (N={len(ids):,})")
    for gt in ['young', 'young_codes', 'dissco', 'cooper']:
        col = f'{prefix}_{gt}_tier'
        if col in cohort.columns:
            counts = (cohort[col].value_counts()
                                  .reindex(TIER_ORDER, fill_value=0))
            pcts   = (counts / counts.sum() * 100).round(1)
            row    = '  |  '.join(f"{t}: {n:,} ({p}%)"
                                   for t, n, p in zip(TIER_ORDER, counts, pcts))
            print(f"    {gt:<12}: {row}")

# Save
cohort.to_csv('/content/cohort_with_GT.csv', index=False)
print(f"\n✅ Saved /content/cohort_with_GT.csv — "
      f"{len(cohort):,} rows × {len(cohort.columns)} columns")

GROUND TRUTH SCORING — ALL INDEX DATES

── TD (46,330 patients) ──────────────────
   Sliced: cond=79,576 | obs=62,595 | med=33,811 | proc=3,742
   Young (lab-aug):
td_young_tier
Baseline T2D              39394
Mild Complications         6425
Moderate Complications      509
Advanced/Critical             2
   DISSCO:
td_dissco_tier
Baseline T2D              30321
Mild Complications        15765
Moderate Complications      244
   Cooper:
td_cooper_tier
Baseline T2D              36812
Moderate Complications     6943
Mild Complications         2557
Advanced/Critical            18

── T0 (46,330 patients) ──────────────────
   Sliced: cond=70,453 | obs=128,848 | med=56,078 | proc=6,784
   Young (lab-aug):
t0_young_tier
Baseline T2D              31866
Mild Complications        11654
Moderate Complications     2787
Advanced/Critical            23
   DISSCO:
t0_dissco_tier
Baseline T2D              26889
Mild Complications        18644
Moderate Complications      795
Advanced/Critical         

In [ ]:
# =============================================================================
# CELL 3 — MOSAIC FROZEN FRAMEWORK GT SCORING
# =============================================================================
# NOTEBOOK : Ground Truth Scoring
# PURPOSE  : Apply the deterministic MOSAIC frozen algorithm across all four
#            index dates and merge results into the cohort CSV alongside the
#            Young, DISSCO, and Cooper GTs.
# INPUTS   : cohort, con_all, obs_all, med_all  (from Cell 1)
#            DRUG_CLASSES                        (defined in Cell 2)
#            cohort_with_GT.csv                  (output of Cell 2)
# OUTPUTS  : cohort — updated with MOSAIC tier + evidence + domain sub-tiers
#            /content/cohort_with_GT.csv  (overwritten with MOSAIC columns added)
# COLUMNS ADDED (per prefix td/t0/t5/t10):
#   {prefix}_mosaic_tier         — Baseline T2D / Mild / Moderate / Advanced/Critical
#   {prefix}_mosaic_evidence     — all domain flags and lab values used
#   {prefix}_mosaic_d1_hba1c     — Domain 1 sub-tier (glycaemic anchor)
#   {prefix}_mosaic_d2_fpg       — Domain 2 sub-tier (fasting glucose fallback)
#   {prefix}_mosaic_d5_comorb    — Domain 5 sub-tier (comorbidities)
#   {prefix}_mosaic_d6_cardrenal — Domain 6 sub-tier (cardiorenal)
#   {prefix}_mosaic_d7_betacell  — Domain 7 sub-tier (pharmacotherapy proxy)
#   {prefix}_mosaic_n_tier3_dom  — number of Tier-3 domains (Rule 0a flag)
# PACKAGES : pandas, numpy
# =============================================================================

import pandas as pd
import numpy as np

# ── LOINC codes used by the MOSAIC frozen framework ──────────────────────────
MOSAIC_LOINC = {
    "hba1c":    "4548-4",     # HbA1c — primary glycaemic anchor
    "fpg":      "1558-6",     # Fasting plasma glucose — secondary glycaemic
    "egfr":     "33914-3",    # eGFR (CKD-EPI 2021)
    "uacr":     "14959-1",    # Urine albumin-to-creatinine ratio
}

# ── SNOMED comorbidity codes used by MOSAIC Domains 5 & 6 ────────────────────
MOSAIC_COMORB = {
    "hypertension":  [59621000, 38341003],
    "dyslipidemia":  [55822004, 13644009, 707534000],
    "cad":           [53741008, 413844008, 194828000],
    "heart_failure": [84114007, 981000124106],
    "stroke":        [230690007, 266257000],
    "pvd":           [698754002, 44808001],
    "ckd":           [127013003, 90781000119102, 46177005, 709044004],
}

# Oral antidiabetic classes for Domain 7 pharmacotherapy proxy
# (DRUG_CLASSES was defined in Cell 2 — reused here)
OAD_CLASSES = {
    "Metformin":    DRUG_CLASSES["Metformin"],
    "Sulfonylurea": DRUG_CLASSES["Sulfonylurea"],
    "Liraglutide":  DRUG_CLASSES.get("Liraglutide",  [897122]),
    "Canagliflozin":DRUG_CLASSES.get("Canagliflozin",[1373463]),
}


# ══════════════════════════════════════════════════════════════════════════════
# MOSAIC FROZEN ALGORITHM
# ══════════════════════════════════════════════════════════════════════════════
def classify_mosaic_frozen(con_slice, obs_slice, med_slice, patient_ids, prefix):
    """
    Deterministic operationalisation of consolidated_framework_v3_T5.

    Rule 0  — final tier = highest tier triggered across all domains.
    Rule 0a — escalate to Advanced/Critical when ANY of:
              (a) ≥2 concurrent Tier-3 domain criteria are met
              (b) eGFR < 15
              (c) insulin dependence + macrovascular disease in ≥2 territories
              (d) HbA1c ≥ 11.0 AND at least one other domain is Tier 3/4
    """

    # ── Lab helpers ───────────────────────────────────────────────────────────
    def lab_agg(loinc_code, agg_func):
        obs_slice["_val"] = pd.to_numeric(obs_slice["VALUE"], errors="coerce")
        sub = (obs_slice[obs_slice["CODE"].astype(str) == str(loinc_code)]
               .dropna(subset=["_val"]).query("_val > 0"))
        return sub.groupby("PATIENT")["_val"].agg(agg_func) if len(sub) else pd.Series(dtype=float)

    # ── Condition / medication set helpers ────────────────────────────────────
    def pts_cond(snomed_codes):
        if not snomed_codes: return set()
        mask = con_slice["CODE"].dropna().astype(float).isin(snomed_codes)
        return set(con_slice[mask]["PATIENT"].unique())

    def pts_med(rxnorm_codes):
        if not rxnorm_codes: return set()
        mask = med_slice["CODE"].dropna().astype(float).isin(rxnorm_codes)
        return set(med_slice[mask]["PATIENT"].unique())

    # ── Pre-compute labs ──────────────────────────────────────────────────────
    hba1c_max = lab_agg(MOSAIC_LOINC["hba1c"], "max")
    fpg_max   = lab_agg(MOSAIC_LOINC["fpg"],   "max")
    uacr_max  = lab_agg(MOSAIC_LOINC["uacr"],  "max")

    # eGFR — use min (worst observed value in window)
    sub_egfr = (obs_slice[obs_slice["CODE"].astype(str) == MOSAIC_LOINC["egfr"]]
                .assign(_val=lambda x: pd.to_numeric(x["VALUE"], errors="coerce"))
                .dropna(subset=["_val"]).query("_val > 0"))
    egfr_min = (sub_egfr.groupby("PATIENT")["_val"].min()
                if len(sub_egfr) else pd.Series(dtype=float))

    # ── Pre-compute condition sets ────────────────────────────────────────────
    htn_pts    = pts_cond(MOSAIC_COMORB["hypertension"])
    dys_pts    = pts_cond(MOSAIC_COMORB["dyslipidemia"])
    cad_pts    = pts_cond(MOSAIC_COMORB["cad"])
    hf_pts     = pts_cond(MOSAIC_COMORB["heart_failure"])
    stroke_pts = pts_cond(MOSAIC_COMORB["stroke"])
    pvd_pts    = pts_cond(MOSAIC_COMORB["pvd"])

    # Macrovascular territories: coronary, cerebrovascular, peripheral
    macro_territories = [cad_pts, stroke_pts, pvd_pts]

    # ── Pre-compute medication sets ───────────────────────────────────────────
    insulin_pts = pts_med(DRUG_CLASSES["Insulin"])
    oad_sets    = {name: pts_med(codes) for name, codes in OAD_CLASSES.items()}

    # ─────────────────────────────────────────────────────────────────────────
    rows = []

    for pid in patient_ids:

        # Per-patient values
        hba1c  = hba1c_max.get(pid, np.nan)
        fpg    = fpg_max.get(pid,   np.nan)
        egfr   = egfr_min.get(pid,  np.nan)
        uacr   = uacr_max.get(pid,  np.nan)

        on_insulin = int(pid in insulin_pts)
        n_oad      = sum(1 for s in oad_sets.values() if pid in s)

        has_htn    = int(pid in htn_pts)
        has_dys    = int(pid in dys_pts)
        has_cad    = int(pid in cad_pts)
        has_hf     = int(pid in hf_pts)
        has_stroke = int(pid in stroke_pts)
        has_pvd    = int(pid in pvd_pts)
        has_cvd    = int(has_cad or has_hf or has_stroke or has_pvd)

        n_macro_territories = sum(1 for s in macro_territories if pid in s)
        has_ascvd  = int(has_cad or has_stroke or has_pvd)

        # ── Domain 1 — HbA1c (primary glycaemic anchor) ──────────────────────
        if pd.notna(hba1c):
            if   hba1c >= 11.0: d1 = 4   # conditional — resolved by Rule 0a
            elif hba1c >=  8.0: d1 = 3
            elif hba1c >=  7.0: d1 = 2
            else:               d1 = 1
        else:
            d1 = None

        # ── Domain 2 — Fasting Plasma Glucose (fallback when HbA1c absent) ───
        if pd.notna(fpg):
            if   fpg > 154: d2 = 3        # conditional — resolved by Rule 0a
            elif fpg >= 126: d2 = 2
            else:            d2 = 1
        else:
            d2 = None

        glyc_tier = d1 if d1 is not None else d2   # HbA1c preferred

        # ── Domain 5 — Comorbid conditions ───────────────────────────────────
        if n_macro_territories >= 2 and on_insulin:
            d5 = 4                           # Tier 4 conditional (Rule 0a c)
        elif has_cvd or (has_htn and has_dys):
            d5 = 3
        elif has_htn or has_dys:
            d5 = 2
        else:
            d5 = 1

        # ── Domain 6 — Cardiorenal ────────────────────────────────────────────
        if pd.notna(egfr):
            if   egfr < 15:  d6 = 4
            elif egfr < 30:  d6 = 4 if (has_ascvd and has_hf) else 3
            elif egfr < 60:  d6 = 2
            else:            d6 = 1
        else:
            d6 = 1

        # UACR overrides
        if pd.notna(uacr):
            if   uacr > 300: d6 = max(d6, 3)   # macroalbuminuria
            elif uacr >= 30: d6 = max(d6, 2)   # microalbuminuria

        # ASCVD + HF without eGFR info
        if has_ascvd and has_hf:
            d6 = max(d6, 3)

        # ── Domain 7 — Beta-cell / pharmacotherapy proxy ─────────────────────
        if on_insulin and n_oad >= 2:
            d7 = 4    # conditional — resolved by Rule 0a
        elif on_insulin or n_oad >= 3:
            d7 = 3
        elif n_oad >= 2:
            d7 = 2
        else:
            d7 = 1

        # ── Rule 0 — collect active domain tiers ─────────────────────────────
        active = {
            "D1_HbA1c":       glyc_tier,
            "D5_Comorb":      d5,
            "D6_Cardiorenal": d6,
            "D7_BetaCell":    d7,
        }
        active = {k: v for k, v in active.items() if v is not None}

        # ── Rule 0a — Advanced/Critical escalation checks ────────────────────
        n_tier3 = sum(1 for v in active.values() if v >= 3)
        egfr_critical   = pd.notna(egfr) and egfr < 15
        insulin_macro   = bool(on_insulin and n_macro_territories >= 2)
        hba1c_t4_cond   = (pd.notna(hba1c) and hba1c >= 11.0 and
                           any(v >= 3 for k, v in active.items() if k != "D1_HbA1c"))

        if n_tier3 >= 2 or egfr_critical or insulin_macro or hba1c_t4_cond:
            final_tier = 4
        else:
            # Cap conditional Tier-4 signals at Tier 3 if Rule 0a not met
            capped     = {k: min(v, 3) for k, v in active.items()}
            final_tier = max(capped.values()) if capped else 1

        # ── Tier label ────────────────────────────────────────────────────────
        TIER_MAP = {
            1: "Baseline T2D",
            2: "Mild Complications",
            3: "Moderate Complications",
            4: "Advanced/Critical",
        }
        tier_label = TIER_MAP[final_tier]

        # ── Evidence string ───────────────────────────────────────────────────
        ev = []
        if pd.notna(hba1c):    ev.append(f"hba1c={hba1c:.1f}%→D1_T{d1 or '?'}")
        if pd.notna(fpg):      ev.append(f"fpg={fpg:.0f}mg/dL→D2_T{d2 or '?'}")
        if pd.notna(egfr):     ev.append(f"egfr={egfr:.1f}mL/min→D6_T{d6}")
        if pd.notna(uacr):     ev.append(f"uacr={uacr:.0f}mg/g")
        if has_htn:            ev.append("HTN")
        if has_dys:            ev.append("dyslip")
        if has_cad:            ev.append("CAD")
        if has_hf:             ev.append("HF")
        if has_stroke:         ev.append("stroke")
        if has_pvd:            ev.append("PVD")
        if on_insulin:         ev.append("insulin")
        if n_oad:              ev.append(f"OAD_classes={n_oad}")
        if n_tier3 >= 2:       ev.append(f"Rule0a_a:n_T3_doms={n_tier3}")
        if egfr_critical:      ev.append("Rule0a_b:eGFR<15")
        if insulin_macro:      ev.append(f"Rule0a_c:insulin+macro={n_macro_territories}terr")
        if hba1c_t4_cond:      ev.append("Rule0a_d:HbA1c>=11+other_T3")

        rows.append({
            "PATIENT":                       pid,
            f"{prefix}_mosaic_tier":         tier_label,
            f"{prefix}_mosaic_evidence":     "; ".join(ev) if ev else "no data",
            f"{prefix}_mosaic_d1_hba1c":     d1,
            f"{prefix}_mosaic_d2_fpg":       d2,
            f"{prefix}_mosaic_d5_comorb":    d5,
            f"{prefix}_mosaic_d6_cardrenal": d6,
            f"{prefix}_mosaic_d7_betacell":  d7,
            f"{prefix}_mosaic_n_tier3_dom":  n_tier3,
        })

    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# SCORE ALL FOUR INDEX DATES
# ══════════════════════════════════════════════════════════════════════════════

eligible_ids = set(cohort["PATIENT"])
ids_t5       = set(cohort.loc[cohort["eligible_t5"]  == 1, "PATIENT"])
ids_t10      = set(cohort.loc[cohort["eligible_t10"] == 1, "PATIENT"])

WINDOWS = [
    ("td",  eligible_ids, "td_cov_start",  "td_cov_end"),
    ("t0",  eligible_ids, "t0_cov_start",  "t0_cov_end"),
    ("t5",  ids_t5,       "t5_cov_start",  "t5_cov_end"),
    ("t10", ids_t10,      "t10_cov_start", "t10_cov_end"),
]

TIER_ORDER = ["Baseline T2D", "Mild Complications",
              "Moderate Complications", "Advanced/Critical"]

print("=" * 60)
print("MOSAIC FROZEN ALGORITHM — ALL INDEX DATES")
print("=" * 60)

for prefix, ids, start_col, end_col in WINDOWS:
    print(f"\n── {prefix.upper()} ({len(ids):,} patients) ──────────────────")

    # Attach patient-specific window dates and slice
    win_cols = cohort[["PATIENT", start_col, end_col]].rename(
        columns={start_col: "WIN_START", end_col: "WIN_END"})

    def slice_w(df, date_col):
        d = df[df["PATIENT"].isin(ids)].merge(win_cols, on="PATIENT", how="inner")
        return d[(d[date_col] >= d["WIN_START"]) & (d[date_col] <= d["WIN_END"])]

    con_sl  = slice_w(con_all,  "START")
    obs_sl  = slice_w(obs_all,  "DATE")
    med_sl  = slice_w(med_all,  "START")

    print(f"   Sliced: cond={len(con_sl):,} | obs={len(obs_sl):,} | med={len(med_sl):,}")

    mosaic_df = classify_mosaic_frozen(con_sl, obs_sl, med_sl, list(ids), prefix)

    # Distribution summary
    counts = (mosaic_df[f"{prefix}_mosaic_tier"]
              .value_counts()
              .reindex(TIER_ORDER, fill_value=0))
    pcts   = (counts / counts.sum() * 100).round(1)
    print(f"   MOSAIC tier distribution:")
    for tier, n, p in zip(TIER_ORDER, counts, pcts):
        print(f"     {tier:<30}  {n:>6,}  ({p:.1f}%)")

    # Merge into cohort (drop existing columns to avoid _x/_y on re-run)
    existing = [c for c in mosaic_df.columns if c in cohort.columns and c != "PATIENT"]
    cohort   = cohort.drop(columns=existing, errors="ignore")
    cohort   = cohort.merge(mosaic_df, on="PATIENT", how="left")

# ── Final summary across all GTs and windows ─────────────────────────────────
print("\n" + "=" * 60)
print("FULL GT SUMMARY — all algorithms × all windows")
print("=" * 60)

for prefix, ids, _, _ in WINDOWS:
    print(f"\n  {prefix.upper()}  (N={len(ids):,})")
    for gt in ["young", "young_codes", "dissco", "cooper", "mosaic"]:
        col = f"{prefix}_{gt}_tier"
        if col not in cohort.columns:
            continue
        counts = cohort[col].value_counts().reindex(TIER_ORDER, fill_value=0)
        pcts   = (counts / counts.sum() * 100).round(1)
        row    = "  |  ".join(
            f"{t.split()[0]}: {n:,} ({p}%)"
            for t, n, p in zip(TIER_ORDER, counts, pcts))
        print(f"    {gt:<12}: {row}")

# ── Save updated cohort ───────────────────────────────────────────────────────
cohort.to_csv("/content/cohort_with_GT.csv", index=False)
print(f"\n✅ Saved /content/cohort_with_GT.csv — "
      f"{len(cohort):,} rows × {len(cohort.columns)} columns")

MOSAIC FROZEN ALGORITHM — ALL INDEX DATES

── TD (46,330 patients) ──────────────────
   Sliced: cond=79,576 | obs=62,595 | med=33,811
   MOSAIC tier distribution:
     Baseline T2D                    32,421  (70.0%)
     Mild Complications              12,297  (26.5%)
     Moderate Complications           1,592  (3.4%)
     Advanced/Critical                   20  (0.0%)

── T0 (46,330 patients) ──────────────────
   Sliced: cond=70,453 | obs=128,848 | med=56,078
   MOSAIC tier distribution:
     Baseline T2D                    31,008  (66.9%)
     Mild Complications               8,925  (19.3%)
     Moderate Complications           6,273  (13.5%)
     Advanced/Critical                  124  (0.3%)

── T5 (41,913 patients) ──────────────────
   Sliced: cond=74,614 | obs=243,139 | med=61,266
   MOSAIC tier distribution:
     Baseline T2D                    25,276  (60.3%)
     Mild Complications               6,226  (14.9%)
     Moderate Complications           9,818  (23.4%)
     Advan